In [ ]:

# This is Open Source Work must be used for good purposes
# NO MASS PROCESSING ,NO STEALING , NO DATA COLLECTION ALLOWED
# THIS IS FULL WORK FOR GOOD

######################### ASR #########################


# import datasets for the dataset
!pip install datasets

# import faster_whisper ASR
# import SpeachRecognition
!pip install SpeechRecognition
import speech_recognition as sr

!pip install whisperx
# import WhisperX
import whisperx
from faster_whisper import WhisperModel

!pip install vosk
# import vosk
from vosk import KaldiRecognizer, Model
# Load the dataset metadata
import polars as pl
import datasets as dsl
from datasets import Audio
import numpy as np
!pip install jiwer
import jiwer

In [ ]:
# ds = dsl.load_dataset("jacktol/ATC-ASR-Dataset")

# Load and explicitly cast the audio column
ds = dsl.load_dataset("jacktol/ATC-ASR-Dataset", split='train', streaming=True)
ds = ds.cast_column("audio", Audio(sampling_rate=16000)) # Force decoding

print(df.head())

# --- 2. Setup Lists for Results ---
references = []
hypotheses = []
max_samples = 50  # Start small to test; ATC data is large!

shape: (5, 3)
┌──────────────────────┬─────────────────────────────────┬─────────────────────────────────┐
│ id                   ┆ audio                           ┆ text                            │
│ ---                  ┆ ---                             ┆ ---                             │
│ str                  ┆ struct[2]                       ┆ str                             │
╞══════════════════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 0023HAQRADETCJDF66RO ┆ {b"RIFF\xa4\xb1\x00\x00WAVEfmt… ┆ SIERRA DELTA MIKE               │
│ 005S1QQUAXJEA93OY8R9 ┆ {b"RIFF\xe4\xbb\x01\x00WAVEfmt… ┆ ONE TWO SEVEN ONE TWO FIVE GOO… │
│ 008AJHCDF6MQ886KORNL ┆ {b"RIFFd6\x02\x00WAVEfmt\x20\x… ┆ CSA ZERO TWO FIVE HEADING THRE… │
│ 00VNAOK94F1OMDT7TFFG ┆ {b"RIFFdw\x02\x00WAVEfmt\x20\x… ┆ TWINSTAR EIGHT EIGHT SEVEN DES… │
│ 01d7425638602ab696a4 ┆ {b"RIFF\xe48\x02\x00WAVEfmt\x2… ┆ ARE CLEARED TO LAND RUNWAY TWO… │
└──────────────────────┴────────────────────────────────

In [ ]:


# here the function we run the dataset on all the samples from the dataset and ecaluate the results to verify each library
def transcribe_hf_audio(dataset_row):
    recognizer = sr.Recognizer()

    # 1. Extract audio data and sampling rate
    # HF datasets return a dict: {'array': np.array, 'sampling_rate': int}
    audio_array = dataset_row['audio']['array']
    sample_rate = dataset_row['audio']['sampling_rate']

    # 2. Convert Float32 (HF standard) to Int16 (sr standard)
    # Most SR engines expect 16-bit PCM
    audio_int16 = (audio_array * 32767).astype(np.int16)
    audio_bytes = audio_int16.tobytes()

    # 3. Create the AudioData object
    # sample_width=2 because Int16 is 2 bytes
    audio_data = sr.AudioData(audio_bytes, sample_rate, sample_width=2)

    # 4. Recognize using Google (or any other sr method)
    try:
        text = recognizer.recognize_google(audio_data)
        return text
    except sr.UnknownValueError:
        return "[Could not understand audio]"
    except sr.RequestError as e:
        return f"[API Error: {e}]"

# --- 3. Iteration Loop ---
for i, sample in enumerate(ds):
    if i >= max_samples:
        break

    # Get the Ground Truth (Reference)
    ref_text = sample['text'].lower()

    # Process Audio for SR
    audio_array = sample['audio']['array']
    # Normalize and convert to int16
    audio_int16 = (audio_array * 32767).astype(np.int16)
    audio_data = sr.AudioData(audio_int16.tobytes(), 16000, sample_width=2)

    try:
        # Get the Prediction (Hypothesis)
        hyp_text = recognizer.recognize_google(audio_data).lower()
    except:
        hyp_text = ""  # If it fails to hear anything, we treat it as an empty string

    references.append(ref_text)
    hypotheses.append(hyp_text)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{max_samples}...")

# --- 4. Calculate Accuracy ---
wer = jiwer.wer(references, hypotheses)
accuracy = (1 - wer) * 100

print(f"\n--- Final Results ---")
print(f"Word Error Rate (WER): {wer:.2%}")
print(f"Overall Accuracy: {accuracy:.2f}%")

Processed 10/50...
Processed 20/50...
Processed 30/50...
Processed 40/50...
Processed 50/50...

--- Final Results ---
Word Error Rate (WER): 100.00%
Overall Accuracy: 0.00%


In [ ]:
# Essential Importing
!pip install datasets
!pip install polars
!pip install SpeechRecognition
!pip install jiwer
!pip install noisereduce
import datasets as ds
import polars as pl
import speech_recognition as sr
import numpy as np
import jiwer
import noisereduce

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.7 MB/s eta 0:00:00


In [ ]:
# time to understand the dataset
df =  ds.load_dataset("jacktol/ATC-ASR-Dataset",split="train")
predictions = []
actualones = []
Right = 0
total = 0
for i in range(0,500):
  x = df.cast_column("audio",ds.Audio(sampling_rate=16000))
  sample = (x[i]["audio"]["array"]* 32767).astype(np.int16).tobytes()
  audiodata = sr.AudioData(sample,sample_width=2,sample_rate=16000)
  recognizer = sr.Recognizer()
  try :
    text = recognizer.recognize_google(audiodata)
  except :
    continue
  # print(text)
  # print(i)
  total = total + 1
  predictions.append(text.lower())
  actualones.append(x[i]["text"].lower())


In [ ]:
print("==================================")
print("==================================")
print("==================================")
print("==================================")
print("==================================")
print("==================================")
WER = jiwer.wer(predictions,actualones)
print(WER)
print("==================================")
print("==================================")
print("==================================")
print("==================================")
print("==================================")
print("==================================")

2.2395437262357416
